In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import pandas as pd

df = pd.read_csv("../dataset/dataset_clean.csv")
df = df[df["rating_label"] != "neutral"].copy()
data = df.drop(columns = ["year", "month", "day", "Reviewer Name", "Country", "rating_label"])
labels = df["rating_label"]

In [2]:
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLm-L6-v2")

documents = []

for idx, text in enumerate(data["Review Text"]):
    try:
        label = labels.iloc[idx]
    except AttributeError:
        label = labels[idx]
    sentiment_label = "positive" if label == 1 else "negative"

    doc = Document(
        page_content = text,
        metadata ={
            "sentiment" : sentiment_label,
            "id" : idx,
        }
    )

    documents.append(doc)

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory = "../database/chroma_db_reviews"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
query = "The room was very dirty and the breakfast was terrible"

results = vector_store.similarity_search(query, k=3)

for i, res in enumerate(results):
    print(res.page_content, res.metadata['sentiment'])

Absolutely horrible. negative
THE WORST SHOPPING EXPERIENCE OF MY ENTIRE LIFE negative
Awful, treat you like dirt. negative
